# Create, evaluate, and deploy an AI agent

This notebook walks you through building, evaluating, and deploying an AI agent that combines retrieval and tool usage. You'll work with a pre-chunked subset of Databricks documentation as your dataset.

This version has been adapted from the original Databricks notebook to run locally in VS Code or Jupyter with a normal Python kernel. Anything that only works inside Databricks (Unity Catalog, Databricks Model Serving, Agent Evaluation, deployment APIs) is called out and replaced with a local equivalent where one exists, or clearly marked as Databricks-only otherwise.

## Setup

Install the packages needed for this lab. Create and activate a virtual environment first (`python -m venv .venv`, then activate it), then run the cell below.

We use `langchain-openai` here as the LLM provider instead of `databricks-langchain`. Swap it for whichever provider you use (Azure OpenAI, Anthropic, local Ollama, etc.) if needed, LangChain has adapters for most of them.

In [ ]:
%pip install -qU mlflow langchain langgraph langchain-openai pydantic scikit-learn pandas requests

## About Unity Catalog

The original lab registers a tool function in Databricks Unity Catalog so it can be discovered and reused across notebooks and served securely. Unity Catalog is a Databricks-only feature, so there is no local equivalent for the catalog and schema registry part.

For this local version, tools are just plain Python functions wrapped with LangChain's `@tool` decorator. You lose cross-workspace tool discovery and governance, but the agent logic and behavior stay the same.

## Create an agent and tools

First, set up the LLM. The original notebook uses `ChatDatabricks` pointed at a Databricks Model Serving endpoint. Locally, we use `ChatOpenAI` instead.

**Fill in your own API key and model name below.** Set the `OPENAI_API_KEY` environment variable (or put it in a `.env` file and load it with `python-dotenv`) instead of hardcoding it in the notebook.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# set OPENAI_API_KEY as an environment variable before running this cell
os.environ.setdefault("OPENAI_API_KEY", "PASTE_YOUR_OPENAI_API_KEY_HERE")

# TODO: replace with the model you want to use
LLM_MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=LLM_MODEL, temperature=0.01)

Next, load the sample Databricks documentation dataset. This is a public file on GitHub, so it loads the same way locally as it does on Databricks.

In [ ]:
import pandas as pd

databricks_docs_url = "https://raw.githubusercontent.com/databricks/genai-cookbook/refs/heads/main/quick_start_demo/chunked_databricks_docs_filtered.jsonl"
parsed_docs_df = pd.read_json(databricks_docs_url, lines=True)
parsed_docs_df.head()

Now create the keyword extraction tool. In the original notebook this function gets registered in Unity Catalog so the agent (and other notebooks) can call it as a governed tool. Locally, we just register it as a normal LangChain tool.

In [ ]:
from langchain_core.tools import tool
from sklearn.feature_extraction.text import TfidfVectorizer


@tool
def tfidf_keywords(text: str) -> list[str]:
    """
    Extracts keywords from the provided text using TF-IDF.

    Args:
        text (string): Input text.
    Returns:
        list[str]: List of extracted keywords in ascending order of importance.
    """

    def extract_keywords(text, top_n=5):
        # fit a fresh vectorizer on just this text
        keyword_vectorizer = TfidfVectorizer(stop_words="english")
        query_tfidf = keyword_vectorizer.fit_transform([text])
        scores = query_tfidf.toarray()[0]
        indices = scores.argsort()[-top_n:][::-1]
        return [
            keyword_vectorizer.get_feature_names_out()[i]
            for i in indices
            if scores[i] > 0
        ]

    return extract_keywords(text)

In [ ]:
print(tfidf_keywords)
tfidf_keywords.invoke({"text": "The quick brown fox jumped over the lazy brown dog."})

Next, build a simple retriever tool over the documentation using TF-IDF similarity. `mlflow.trace` works the same way locally as it does on Databricks, it's part of open source MLflow, not a Databricks-only feature.

In [ ]:
from typing import Any

import mlflow
from langchain_core.tools import tool
from sklearn.feature_extraction.text import TfidfVectorizer

documents = parsed_docs_df
doc_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = doc_vectorizer.fit_transform(documents["content"])


@tool
@mlflow.trace(name="LittleIndex", span_type=mlflow.entities.SpanType.RETRIEVER)
def find_relevant_documents(query: str, top_n: int = 5) -> list[dict[str, Any]]:
    """gets relevant documents for the query"""
    query_tfidf = doc_vectorizer.transform([query])
    similarities = (tfidf_matrix @ query_tfidf.T).toarray().flatten()
    ranked_docs = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

    result = []
    for idx, score in ranked_docs[:top_n]:
        row = documents.iloc[idx]
        content = row["content"]
        doc_entry = {
            "page_content": content,
            "metadata": {
                "doc_uri": row["doc_uri"],
                "score": score,
            },
        }
        result.append(doc_entry)
    return result

Now define the LangGraph agent loop. This ties the LLM and tools together with a simple routing function: keep calling tools until the model stops asking for one, then return.

`ChatAgentState` and `ChatAgentToolNode` come from `mlflow.langchain.chat_agent_langgraph`, which is part of open source MLflow, so this code runs unchanged locally.

In [ ]:
from typing import Optional, Sequence, Union

from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    agent_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    def routing_logic(state: ChatAgentState):
        last_message = state["messages"][-1]
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if agent_prompt:
        system_message = {"role": "system", "content": agent_prompt}
        preprocessor = RunnableLambda(
            lambda state: [system_message] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        routing_logic,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()

Try the agent out. `mlflow.langchain.autolog()` turns on automatic tracing for LangChain and LangGraph calls, this works the same locally as it does on Databricks.

In [ ]:
import mlflow

mlflow.langchain.autolog()

agent = create_tool_calling_agent(llm, tools=[tfidf_keywords, find_relevant_documents])
agent.invoke({"messages": [{"role": "user", "content": "What are the keywords for the sentence: 'the quick brown fox jumped over the lazy brown dog'?"}]})

## Wrap the agent as an MLflow ChatAgent

`mlflow.pyfunc.ChatAgent` is the standard interface MLflow uses for conversational agents. This is open source MLflow, so it works the same locally as on Databricks.

In [ ]:
from typing import Any, Optional

from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)


class DocsAgent(ChatAgent):
    def __init__(self, agent):
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        # ChatAgent has a built-in helper to convert framework messages to plain dicts
        request = {"messages": self._convert_messages_to_dict(messages)}
        output = self.agent.invoke(request)
        return ChatAgentResponse(**output)

In [ ]:
AGENT = DocsAgent(agent=agent)
AGENT.predict({"messages": [{"role": "user", "content": "What is DLT in Databricks?"}]})

## Make the agent configurable

Instead of hardcoding the endpoint and prompt, wrap them in a config dict. This makes it easy to swap models or tweak the prompt without changing code. `ModelConfig` is open source MLflow and works the same locally.

In [ ]:
from mlflow.models import ModelConfig

baseline_config = {
    "model_name": LLM_MODEL,
    "temperature": 0.01,
    "max_tokens": 1000,
    "system_prompt": """You are a helpful assistant that answers questions about Databricks. Questions unrelated to Databricks are irrelevant.

You answer questions using a set of tools. If needed, you ask the user follow-up questions to clarify their request.
""",
}


class DocsAgent(ChatAgent):
    def __init__(self, config, tools):
        self.config = ModelConfig(development_config=config)
        self.tools = tools
        self.agent = self._build_agent_from_config()

    def _build_agent_from_config(self):
        temperature = self.config.get("temperature")
        max_tokens = self.config.get("max_tokens")
        system_prompt = self.config.get("system_prompt")
        model_name = self.config.get("model_name")

        llm = ChatOpenAI(model=model_name, temperature=temperature, max_tokens=max_tokens)
        agent = create_tool_calling_agent(llm, tools=self.tools, agent_prompt=system_prompt)
        return agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}
        output = self.agent.invoke(request)
        return ChatAgentResponse(**output)


agent_tools = [tfidf_keywords, find_relevant_documents]
agent = DocsAgent(baseline_config, agent_tools)
agent.predict({"messages": [{"role": "user", "content": "What is DLT"}]})

## Package the agent as a standalone script

To log and deploy the agent with MLflow, it helps to have all the agent code in one file. The `%%writefile` magic works fine in Jupyter and VS Code, it just writes a `.py` file to disk.

This script drops the Databricks-only imports (`databricks_langchain`, `UCFunctionToolkit`, `DatabricksFunctionClient`) and uses plain LangChain tools and `ChatOpenAI` instead.

In [ ]:
%%writefile getting_started_agent.py
from typing import Any, Optional, Sequence, Union

import mlflow
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool, tool
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.models import ModelConfig
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)
from sklearn.feature_extraction.text import TfidfVectorizer

databricks_docs_url = "https://raw.githubusercontent.com/databricks/genai-cookbook/refs/heads/main/quick_start_demo/chunked_databricks_docs_filtered.jsonl"
parsed_docs_df = pd.read_json(databricks_docs_url, lines=True)

documents = parsed_docs_df
doc_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = doc_vectorizer.fit_transform(documents["content"])


@tool
@mlflow.trace(name="LittleIndex", span_type=mlflow.entities.SpanType.RETRIEVER)
def find_relevant_documents(query: str, top_n: int = 5) -> list[dict[str, Any]]:
    """gets relevant documents for the query"""
    query_tfidf = doc_vectorizer.transform([query])
    similarities = (tfidf_matrix @ query_tfidf.T).toarray().flatten()
    ranked_docs = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

    result = []
    for idx, score in ranked_docs[:top_n]:
        row = documents.iloc[idx]
        content = row["content"]
        doc_entry = {
            "page_content": content,
            "metadata": {
                "doc_uri": row["doc_uri"],
                "score": score,
            },
        }
        result.append(doc_entry)
    return result


@tool
def tfidf_keywords(text: str) -> list[str]:
    """
    Extracts keywords from the provided text using TF-IDF.

    Args:
        text (string): Input text.
    Returns:
        list[str]: List of extracted keywords in ascending order of importance.
    """

    def extract_keywords(text, top_n=5):
        keyword_vectorizer = TfidfVectorizer(stop_words="english")
        query_tfidf = keyword_vectorizer.fit_transform([text])
        scores = query_tfidf.toarray()[0]
        indices = scores.argsort()[-top_n:][::-1]
        return [
            keyword_vectorizer.get_feature_names_out()[i]
            for i in indices
            if scores[i] > 0
        ]

    return extract_keywords(text)


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[ToolNode, Sequence[BaseTool]],
    agent_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    def routing_logic(state: ChatAgentState):
        last_message = state["messages"][-1]
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if agent_prompt:
        system_message = {"role": "system", "content": agent_prompt}
        preprocessor = RunnableLambda(
            lambda state: [system_message] + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)
        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        routing_logic,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class DocsAgent(ChatAgent):
    def __init__(self, config, tools):
        # when this agent is deployed, this config is replaced with the config
        # passed to mlflow.pyfunc.log_model(model_config=...)
        self.config = ModelConfig(development_config=config)
        self.tools = tools
        self.agent = self._build_agent_from_config()

    def _build_agent_from_config(self):
        llm = ChatOpenAI(
            model=self.config.get("model_name"),
            temperature=self.config.get("temperature"),
            max_tokens=self.config.get("max_tokens"),
        )
        agent = create_tool_calling_agent(
            llm,
            tools=self.tools,
            agent_prompt=self.config.get("system_prompt"),
        )
        return agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}
        output = self.agent.invoke(request)
        return ChatAgentResponse(**output)


# TODO: replace with the model you want to use
LLM_MODEL = "gpt-4o-mini"

baseline_config = {
    "model_name": LLM_MODEL,
    "temperature": 0.01,
    "max_tokens": 1000,
    "system_prompt": """You are a helpful assistant that answers questions about Databricks. Questions unrelated to Databricks are irrelevant.

You answer questions using a set of tools. If needed, you ask the user follow-up questions to clarify their request.
""",
}

tools = [find_relevant_documents, tfidf_keywords]

AGENT = DocsAgent(baseline_config, tools)
mlflow.models.set_model(AGENT)

A couple of notes on what changed in the script above compared to the original:

- Dropped `databricks_langchain`, `DatabricksFunctionClient`, `UCFunctionToolkit`, and `set_uc_function_client`. There's no Unity Catalog locally, so `tfidf_keywords` and `find_relevant_documents` are just included directly as LangChain tools.
- Swapped `ChatDatabricks` for `ChatOpenAI`.
- Removed the `tool_list` catalog wildcard config key since there's no UC function registry to look tools up from.

Run the cell above once to create `getting_started_agent.py` in your working directory.

The original notebook calls `dbutils.library.restartPython()` here to restart the Python process on the Databricks cluster after writing the file, so the new module can be imported cleanly. `dbutils` doesn't exist outside Databricks.

Locally, you don't need a hard process restart, just re-run the import cell below. If you've already imported `getting_started_agent` earlier in this session and changed the file, restart the Jupyter kernel (Kernel > Restart) so the import picks up the new version.

In [ ]:
from getting_started_agent import AGENT

AGENT.predict({"messages": [{"role": "user", "content": "What is DLT"}]})

## Log the agent with MLflow

Log the agent as an MLflow model so it has a reproducible, versioned artifact you can reload or serve later.

The original notebook also builds a `resources` list of `DatabricksServingEndpoint` and `DatabricksFunction` objects. Those describe Databricks-managed resources (model serving endpoints, Unity Catalog functions) so Databricks can wire up authentication when it deploys the model. There's no equivalent concept locally since you're not deploying to Databricks, so this step is skipped.

In [ ]:
import mlflow
from getting_started_agent import baseline_config, tools

with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        python_model="getting_started_agent.py",
        artifact_path="agent",
        model_config=baseline_config,
        pip_requirements=[
            "mlflow",
            "langchain",
            "langgraph",
            "langchain-openai",
            "pydantic",
            "scikit-learn",
            "pandas",
        ],
        input_example={
            "messages": [{"role": "user", "content": "What is lakehouse monitoring?"}]
        },
    )

print(model_info.model_uri)

## Build an evaluation set

The original notebook uses `databricks.agents.evals.generate_evals_df`, a Databricks-only tool that uses an LLM to auto-generate synthetic question/answer pairs from your docs. There's no open source equivalent that ships with MLflow.

Below is a small local replacement that does the same job: it prompts your LLM to generate a handful of realistic questions from a sample of the docs. It's simpler than the Databricks version (no persona/guideline templating engine) but keeps the same purpose, building an eval set you can run your agent against.

In [ ]:
import json
import random

agent_description = """
The agent is a RAG chatbot that answers questions about Databricks. Questions unrelated to Databricks are irrelevant.
"""

question_guidelines = """
User personas:
- A developer who is new to the Databricks platform
- An experienced, highly technical data scientist or data engineer

Example questions:
- What API lets me parallelize operations over rows of a delta table?
- Which cluster settings will give me the best performance when using Spark?

Additional guidelines:
- Questions should be succinct and human-like
"""

num_evals = 10  # keep this small locally to control API cost
sample_docs = parsed_docs_df.sample(n=min(50, len(parsed_docs_df)), random_state=42)

eval_prompt = f"""{agent_description}

{question_guidelines}

Generate {num_evals} realistic user questions that could be answered using the documentation below.
Return only a JSON array of strings, no other text.

Documentation excerpts:
{chr(10).join(sample_docs["content"].str.slice(0, 500).tolist())}
"""

response = llm.invoke(eval_prompt)
questions = json.loads(response.content)

evals = pd.DataFrame({"inputs": [{"messages": [{"role": "user", "content": q}]} for q in questions]})
evals.head()

## Define a custom scorer

This checks that the agent actually used both the keyword tool and the retriever tool while answering, the same check as in the original notebook. `mlflow.genai.scorers` (open source MLflow's evaluation API) is the modern replacement for the Databricks-only `databricks.agents.evals.metric` decorator, the logic below is functionally the same.

In [ ]:
from mlflow.genai.scorers import scorer


@scorer
def uses_keywords_and_retriever(trace):
    retriever_spans = trace.search_spans(span_type="RETRIEVER")
    keyword_tool_spans = trace.search_spans(name="tfidf_keywords")
    return len(keyword_tool_spans) > 0 and len(retriever_spans) > 0

## Run the evaluation

The original notebook uses `mlflow.evaluate(..., model_type="databricks-agent")`, which activates Databricks-only Mosaic AI Agent Evaluation (built-in judges for groundedness, relevance, safety, etc. hosted on Databricks). That specific model type only works inside Databricks, so it's marked here as Databricks-specific and not reproduced.

What we can do locally is run the open source `mlflow.genai.evaluate()` with our custom scorer above. This checks tool usage the same way the original notebook's `uses_keywords_and_retriever` metric did.

Note: this cell calls your LLM once per eval row, so it can hit provider rate limits on a free tier. Reduce `num_evals` above if you hit rate limits.

In [ ]:
from getting_started_agent import AGENT as agent_for_eval


def predict_fn(messages):
    return agent_for_eval.predict({"messages": messages})


with mlflow.start_run(run_name="my_agent"):
    eval_results = mlflow.genai.evaluate(
        data=evals,
        predict_fn=predict_fn,
        scorers=[uses_keywords_and_retriever],
    )

eval_results

## Deploy the agent (Databricks-specific)

The original notebook registers the model to the Databricks Unity Catalog model registry and deploys it with `databricks.agents.deploy`, which creates a Databricks Model Serving endpoint plus a review app for stakeholder feedback. This whole step is Databricks-only: there is no Unity Catalog, no Agent Framework review app, and no managed serving endpoint outside Databricks.

If you want to actually deploy this on Databricks later, the original code for reference looks like this:

```python
import mlflow
from databricks import agents

mlflow.set_registry_uri("databricks-uc")

catalog = "your_catalog"
schema = "your_schema"
UC_MODEL_NAME = f"{catalog}.{schema}.getting_started_agent"

uc_registered_model_info = mlflow.register_model(
    model_uri=model_info.model_uri, name=UC_MODEL_NAME
)
deployment_info = agents.deploy(
    UC_MODEL_NAME, uc_registered_model_info.version, deploy_feedback_model=False
)
```

For a local equivalent, MLflow can serve the model you logged earlier as a REST API on your own machine using its built-in scoring server. This gives you an `/invocations` endpoint you can call with `curl` or any HTTP client, similar in spirit to a serving endpoint, just without Unity Catalog governance, the review app, or managed hosting.

Run this from a terminal, not from inside the notebook (it starts a long-running server):

```bash
mlflow models serve -m "<paste model_info.model_uri here>" --env-manager local -p 5000
```

Then test it with:

```bash
curl -X POST http://127.0.0.1:5000/invocations \
  -H "Content-Type: application/json" \
  -d '{"inputs": [{"role": "user", "content": "What is DLT"}]}'
```